In [1]:
!pip install pyspark

In [2]:
# ========================================
# IMPORT LIBRARIES
# ========================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, expr, size, length, trim, lower,
    year, month, dayofmonth, coalesce, when, count
)
from pyspark.sql.types import IntegerType, LongType
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ========================================
# INITIALIZE SPARK SESSION
# ========================================
print("Initializing Spark Session...")
spark = SparkSession.builder \
    .appName("YouTubeAnalytics") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark initialized successfully!\n")

Initializing Spark Session...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/14 14:21:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark initialized successfully!



In [4]:
# ========================================
# LOAD RAW DATA
# ========================================
print("Loading raw data...")
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .csv("/kaggle/input/minhminh/raw_data.csv")

print(f"Loaded: {df.count()} rows, {len(df.columns)} columns\n")

Loading raw data...


Loaded: 38916 rows, 16 columns



In [5]:
# ========================================
# EXPLORE DATA STRUCTURE
# ========================================
print("=" * 60)
print("DATA STRUCTURE")
print("=" * 60)
df.printSchema()
print("\nSample data:")
df.show(5, truncate=50)


DATA STRUCTURE
root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: boolean (nullable = true)
 |-- ratings_disabled: boolean (nullable = true)
 |-- video_error_or_removed: boolean (nullable = true)
 |-- description: string (nullable = true)


Sample data:
+-----------+-------------+--------------------------------------------------+--------------------------+---------------+-------------------+--------------------------------------------------+--------+------+--------+-------------+-----------------------------------

In [6]:
# ========================================
# CHECK MISSING VALUES
# ========================================
print("\n" + "=" * 60)
print("MISSING VALUES ANALYSIS")
print("=" * 60)
null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
])
null_counts.show(vertical=True)


MISSING VALUES ANALYSIS


-RECORD 0---------------------
 video_id               | 110 
 trending_date          | 110 
 title                  | 110 
 channel_title          | 110 
 category_id            | 110 
 publish_time           | 110 
 tags                   | 110 
 views                  | 110 
 likes                  | 110 
 dislikes               | 110 
 comment_count          | 110 
 thumbnail_link         | 110 
 comments_disabled      | 110 
 ratings_disabled       | 110 
 video_error_or_removed | 110 
 description            | 670 



In [7]:
# ========================================
# DROP UNUSED COLUMNS
# ========================================
print("\nRemoving unnecessary columns...")
cols_to_drop = ['thumbnail_link', 'comments_disabled', 
                'ratings_disabled', 'video_error_or_removed']
df = df.drop(*cols_to_drop)
print(f"Remaining columns: {len(df.columns)}")


Removing unnecessary columns...
Remaining columns: 12


In [8]:
# ========================================
# REMOVE EMPTY ROWS
# ========================================
print("\nRemoving completely empty rows...")
before = df.count()
df = df.dropna(how='all')
removed = before - df.count()
print(f"Removed: {removed} empty rows")



Removing completely empty rows...


Removed: 110 empty rows


In [9]:
# ========================================
# FILTER VALID TRENDING_DATE
# ========================================
print("\nFiltering valid trending_date format...")
before = df.count()
df = df.filter(col("trending_date").rlike(r"^\d{2}\.\d{2}\.\d{2}$"))
removed = before - df.count()
print(f"Removed: {removed} rows with invalid date format")


Filtering valid trending_date format...


Removed: 0 rows with invalid date format


In [10]:
# ========================================
# REMOVE CORRUPTED VIDEO_ID
# ========================================
print("\nRemoving corrupted video_id...")
before = df.count()
df = df.filter(
    (col("video_id").isNotNull()) & 
    (col("video_id") != "#NAME?") &
    (length(col("video_id")) > 5)
)
removed = before - df.count()
print(f"Removed: {removed} invalid video_id records")


Removing corrupted video_id...


Removed: 347 invalid video_id records


In [11]:
# ========================================
# FILL MISSING DESCRIPTIONS
# ========================================
print("\nFilling missing descriptions...")
df = df.withColumn(
    "description",
    coalesce(col("description"), lit("No description available"))
)
print("Missing descriptions filled with default text")


Filling missing descriptions...
Missing descriptions filled with default text


In [12]:
# ========================================
# CONVERT DATETIME FIELDS
# ========================================
print("\nConverting datetime fields...")
df = df.withColumn(
    "trending_date",
    expr("to_timestamp(trending_date, 'yy.dd.MM')")
)

df = df.withColumn(
    "publish_time",
    expr("to_timestamp(publish_time, \"yyyy-MM-dd'T'HH:mm:ss.SSS'Z'\")")
)
print("Datetime conversion completed")


Converting datetime fields...
Datetime conversion completed


In [13]:
# ========================================
# EXTRACT DATE COMPONENTS
# ========================================
print("\nExtracting date components...")
df = df.withColumn("trending_year", year(col("trending_date"))) \
       .withColumn("trending_month", month(col("trending_date"))) \
       .withColumn("trending_day", dayofmonth(col("trending_date"))) \
       .withColumn("publish_year", year(col("publish_time"))) \
       .withColumn("publish_month", month(col("publish_time")))
print("Date components extracted: year, month, day")



Extracting date components...
Date components extracted: year, month, day


In [14]:
# ========================================
# CONVERT NUMERIC COLUMNS
# ========================================
print("\nConverting and cleaning numeric columns...")
numeric_cols = ['views', 'likes', 'dislikes', 'comment_count']

for col_name in numeric_cols:
    df = df.withColumn(col_name, col(col_name).cast(LongType()))
    df = df.withColumn(
        col_name,
        when(col(col_name).isNull() | (col(col_name) < 0), 0)
        .otherwise(col(col_name))
    )
print(f"Converted columns: {', '.join(numeric_cols)}")



Converting and cleaning numeric columns...
Converted columns: views, likes, dislikes, comment_count


In [15]:
# ========================================
# PROCESS TAGS FIELD
# ========================================
print("\nProcessing tags field...")
df = df.withColumn(
    "tags",
    when(col("tags") == "[none]", lit("")).otherwise(col("tags"))
)

df = df.withColumn(
    "tags",
    expr("SPLIT(REGEXP_REPLACE(tags, '\"', ''), '\\\\|')")
)

df = df.withColumn("tag_count", size(col("tags")))
print("Tags converted to array and counted")


Processing tags field...
Tags converted to array and counted


In [16]:
# ========================================
# CREATE ENGAGEMENT METRICS
# ========================================
print("\nCreating engagement metrics...")

# Engagement rate: (likes + dislikes + comments) / views * 100
df = df.withColumn(
    "engagement_rate",
    expr("ROUND((likes + dislikes + comment_count) / NULLIF(views, 0) * 100, 2)")
)

# Like ratio: likes / (likes + dislikes) * 100
df = df.withColumn(
    "like_ratio",
    expr("ROUND(likes / NULLIF(likes + dislikes, 0) * 100, 2)")
)

# Days from publish to trending
df = df.withColumn(
    "days_to_trend",
    expr("DATEDIFF(trending_date, publish_time)")
)
print("Created metrics: engagement_rate, like_ratio, days_to_trend")


Creating engagement metrics...
Created metrics: engagement_rate, like_ratio, days_to_trend


In [17]:
# ========================================
# REMOVE DUPLICATES
# ========================================
print("\nRemoving duplicate videos...")
before = df.count()
df = df.orderBy(col("trending_date").desc()).dropDuplicates(["video_id"])
removed = before - df.count()
print(f"Removed: {removed} duplicate records")



Removing duplicate videos...


Removed: 35229 duplicate records


In [18]:
# ========================================
# VALIDATE FINAL DATA
# ========================================
print("\n" + "=" * 60)
print("FINAL DATA VALIDATION")
print("=" * 60)
print(f"Final dataset: {df.count()} rows, {len(df.columns)} columns")

critical_cols = ['video_id', 'trending_date', 'title', 'views']
print("\nNull check for critical columns:")
df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in critical_cols
]).show()



FINAL DATA VALIDATION


Final dataset: 3230 rows, 21 columns

Null check for critical columns:


+--------+-------------+-----+-----+
|video_id|trending_date|title|views|
+--------+-------------+-----+-----+
|       0|            0|    0|    0|
+--------+-------------+-----+-----+



In [19]:
# ========================================
# PREVIEW RESULTS
# ========================================
print("\n" + "=" * 60)
print("PREVIEW PROCESSED DATA")
print("=" * 60)
df.select(
    "video_id", "title", "channel_title", 
    "views", "likes", "engagement_rate", "tag_count"
).show(10, truncate=40)



PREVIEW PROCESSED DATA


+-----------+----------------------------------------+---------------------+-------+------+---------------+---------+
|   video_id|                                   title|        channel_title|  views| likes|engagement_rate|tag_count|
+-----------+----------------------------------------+---------------------+-------+------+---------------+---------+
|-3VBPAZPTQI|NEW YEAR'S EVE MAKEUP TUTORIAL | JAMI...|      Jamie Genevieve| 404690| 16002|           4.24|       13|
|-43MBOJnVks|       RAMPAGE - OFFICIAL TRAILER 2 [HD]|Warner Bros. Pictures|1758599| 16188|           1.09|       29|
|-5WBCrazSfg|Neymar will win the Ballon d'Or | Phi...|            Soccer AM| 345486|  6810|           2.21|       30|
|-5aaJJQFvOg|Havana - swing cover | dodie feat. FL...|          doddleoddle|1770509|163869|           9.82|        8|
|-7tSTUR7FG0|   NCT U 엔시티 유 'BOSS' Dance Practice|               SMTOWN|4651793|251125|           5.68|       49|
|-8X32zNup1o|Joe Rogan Experience #1119 - Howard B...|      

In [20]:
# ========================================
# STATISTICS SUMMARY
# ========================================
print("\n" + "=" * 60)
print("STATISTICS SUMMARY")
print("=" * 60)
df.select("views", "likes", "engagement_rate", "days_to_trend") \
  .summary("count", "mean", "min", "max", "stddev") \
  .show()



STATISTICS SUMMARY


+-------+-------------------+------------------+------------------+------------------+
|summary|              views|             likes|   engagement_rate|     days_to_trend|
+-------+-------------------+------------------+------------------+------------------+
|  count|               3230|              3230|              3230|              3230|
|   mean|  4844860.292879257|100143.90959752322|3.6129287925696625|  47.9030959752322|
|    min|               1014|                 0|               0.0|                 0|
|    max|          424538912|           5613827|             22.52|              3990|
| stddev|1.936832504408682E7|315710.79470671917| 2.932176145909684|268.77467774751045|
+-------+-------------------+------------------+------------------+------------------+



In [22]:
# ========================================
# SAVE PROCESSED DATA
# ========================================
print("\nSaving processed data...")
output_path = './data/processed_youtube_data.csv'

# Convert array column to string for CSV export
df_to_save = df.withColumn(
    "tags",
    expr("array_join(tags, '|')")
)

df_to_save.coalesce(1).write \
  .mode('overwrite') \
  .option('header', 'true') \
  .csv(output_path)

print(f"Data saved to: {output_path}")
print("Processing completed successfully!")


Saving processed data...


Data saved to: ./data/processed_youtube_data.csv
Processing completed successfully!


In [23]:

# ========================================
# STOP SPARK SESSION
# ========================================
print("\nStopping Spark session...")
spark.stop()
print("Done!")


Stopping Spark session...
Done!
